# Smart microscopy 3: adaptive imaging

Spend acquisition time only where it is worth it. Three patterns:

- **A, sample-aware exploration.** Instead of rastering a fixed rectangle, start on the sample
  and grow outwards. A tile's neighbours are visited only if the tile contains sample. On a
  section, colony or organoid, this images the object and its border, not the empty glass around
  it.
- **B, detect and zoom in.** Find objects in the exploration tiles. Rank them, centre each one,
  optionally autofocus, and take a "good" image (other exposure or illumination) only of those.
- **C, event-driven time-lapse.** Watch a field of view at a slow rate and switch to a fast burst
  as soon as something changes.

**Safety.** X/Y moves are relative and stay inside a box around the start. Z moves only through
ImSwitch's autofocus (B, off by default), which scans a limited range around the current focus.

## 1 Connect

In [ ]:
import json, time, datetime
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
from IPython.display import display

import imswitchclient.ImSwitchClient as imc
from imswitchclient import smart

# None = $IMSWITCH_API_URL, set by ImSwitch for the notebooks it serves itself.
# From another computer use the browser address, e.g. "http://192.168.178.76/imswitch/api"
IMSWITCH_URL = None

client = imc.ImSwitchClient(IMSWITCH_URL)
print("Connected to:", client.base_uri)
if not hasattr(client, "autofocusManager"):
    raise ImportError("imswitchclient is too old (no smart helpers) - update it, e.g. pip install -U "
                      "https://github.com/openUC2/imswitchclient/archive/refs/heads/main.zip")

STAGE = client.positionersManager.allStageNames[0]
SPEED = 5000                                   # stage speed for all moves
snap = smart.make_snap(client)                 # -> float32 grayscale frame
move = smart.make_relative_mover(client, STAGE, speed=SPEED)   # move(dx_um, dy_um), blocking
WORK_DIR = Path.home() / "smart_microscopy" / "adaptive"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Stage:", STAGE, client.positionersManager.getPositionerPositions()[STAGE])
print("Light sources:", client.lasersManager.getLaserNames())

## 2 Light and a first look

Switch on the illumination you image with. `ILLUMINATION = None` leaves it as it is (e.g. when
you set it up in the ImSwitch UI).

In [ ]:
ILLUMINATION = {"type": "laser", "name": "LED", "value": 1000}
# LED matrix instead:  ILLUMINATION = {"type": "ledmatrix", "intensity": 128}

def light(on=True):
    if ILLUMINATION is None:
        return
    if ILLUMINATION["type"] == "ledmatrix":
        if on:
            client.ledMatrixManager.setAllLED(state=1, intensity=ILLUMINATION["intensity"])
        else:
            client.ledMatrixManager.setAllLEDOff()
    elif on:
        client.lasersManager.setLaserOn(ILLUMINATION["name"], ILLUMINATION["value"])
    else:
        client.lasersManager.setLaserOff(ILLUMINATION["name"])

light(True)
time.sleep(0.5)
frame = snap()
H, W = frame.shape
print(f"frame {W} x {H} px, range {frame.min():.0f}..{frame.max():.0f}")
plt.figure(figsize=(6, 4.5)); plt.imshow(frame, cmap="gray"); plt.axis("off"); plt.show()

## 3 Camera ↔ stage calibration

Closed-loop moves convert "the object is 40 px left of the centre" into a stage move. The notebook
measures that conversion itself. It moves +X and +Y by a few percent of the field of view and finds
the image shift by phase correlation. The measurement includes camera rotation, mirroring and axis
signs, so the loop cannot run the wrong way. The stage returns to where it started.

Use a structured area of the sample. Recalibrate after changing objective, binning or camera. The
result is cached in `WORK_DIR`.

In [ ]:
RECALIBRATE = False
CAL_FILE = WORK_DIR / "stage_camera_calibration.json"


def pixel_size_guess():
    """µm/px from ImSwitch's pixel calibration or objective, 1.0 if neither is known"""
    try:
        r = client.get_json("/PixelCalibrationController/getPixelSizeUm")
        if r.get("pixelSizeUm"):
            return float(r["pixelSizeUm"])
    except Exception:
        pass
    try:
        return float(client.objectiveController.getStatus()["pixelsize"])
    except Exception:
        return 1.0


cal = None
if CAL_FILE.exists() and not RECALIBRATE:
    cal = smart.StageCameraCalibration.load(CAL_FILE)
    if cal.details.get("frame_shape") != [H, W]:
        print("cached calibration was made at another frame size - recalibrating")
        cal = None
if cal is None:
    step_um = 0.15 * min(H, W) * pixel_size_guess()     # ~15 % of the field of view
    print(f"calibrating with {step_um:.1f} µm steps ...")
    cal = smart.StageCameraCalibration.measure(snap, move, step_um=step_um, settle_s=0.3)
    cal.save(CAL_FILE)
print(cal)
print("details:", {k: v for k, v in cal.details.items() if k != "frame_shape"})
FOV_UM = (H * cal.pixel_size_um, W * cal.pixel_size_um)
print(f"field of view ≈ {FOV_UM[1]:.0f} x {FOV_UM[0]:.0f} µm")

## 4 A: What does empty look like?

A tile counts as sample when enough of it differs from empty glass. The notebook needs one look at
an empty area, taken with the same exposure and illumination as the scan. Either move to empty
glass in the UI and leave `EMPTY_OFFSET_UM = None`, or give a relative offset to an empty spot;
the stage then goes there and comes back. Comparing pixel by pixel cancels vignetting and fixed
background patterns. It works for samples brighter than the background (fluorescence, darkfield)
as well as darker ones (brightfield).

In [ ]:
EMPTY_OFFSET_UM = None        # e.g. (-2000, 0): empty glass 2 mm to the left of here

if EMPTY_OFFSET_UM is not None:
    move(*EMPTY_OFFSET_UM)
try:
    empty_ref = smart.EmptyFieldReference([snap() for _ in range(3)])
finally:
    if EMPTY_OFFSET_UM is not None:
        move(-EMPTY_OFFSET_UM[0], -EMPTY_OFFSET_UM[1])
print(f"empty-field noise: {empty_ref.spread:.2f} counts")

In [ ]:
# How much of the current field differs from empty glass (0..1). Run on the sample and on its
# border to choose COVERAGE_THRESHOLD.
frame = snap()
print(f"sample coverage here: {empty_ref.coverage(frame):.3f}")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].imshow(frame, cmap="gray"); axes[1].imshow(empty_ref.difference_mask(frame))
axes[1].set_title("differs from empty")
for a in axes: a.axis("off")
plt.show()

In [ ]:
COVERAGE_THRESHOLD = 0.05     # a tile is "sample" when this fraction of it differs from empty
OVERLAP = 0.15
MAX_TILES = 40
BOX_HALF_WIDTH_UM = 3000
DIAGONALS = False             # also grow into diagonal neighbours

## 5 A: Explore

Breadth-first from the current tile. Tiles are on a grid along the stage axes. The map shows the
sample coverage of every visited tile (sample tiles are outlined), next to a mosaic of what was imaged.

In [ ]:
from collections import deque

x0, y0 = smart.get_xy(client, STAGE)
box = smart.SafetyBox(x0, y0, BOX_HALF_WIDTH_UM, max_step_um=1e9)
pitch = (1 - OVERLAP) * min(FOV_UM)                    # µm between tile centres
neigh = [(1, 0), (-1, 0), (0, 1), (0, -1)] + ([(1, 1), (1, -1), (-1, 1), (-1, -1)] if DIAGONALS else [])

tiles = {}                    # (i, j) -> {"pos", "score", "frame"}
queue, queued = deque([(0, 0)]), {(0, 0)}
here = (x0, y0)
handle = display(None, display_id=True)


def draw_map():
    keys = list(tiles)
    I = np.array([k[0] for k in keys]); J = np.array([k[1] for k in keys])
    grid = np.full((J.max() - J.min() + 1, I.max() - I.min() + 1), np.nan)
    for (i, j), t in tiles.items():
        grid[j - J.min(), i - I.min()] = t["score"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    im = axes[0].imshow(grid, cmap="viridis", origin="lower",
                        extent=(I.min() - .5, I.max() + .5, J.min() - .5, J.max() + .5))
    for (i, j), t in tiles.items():
        if t["score"] >= COVERAGE_THRESHOLD:
            axes[0].add_patch(plt.Rectangle((i - .5, j - .5), 1, 1, fill=False, ec="w", lw=1.5))
    axes[0].set_title(f"{len(tiles)} tiles, {sum(t['score'] >= COVERAGE_THRESHOLD for t in tiles.values())} with sample")
    axes[0].set_xlabel("tile X"); axes[0].set_ylabel("tile Y")
    fig.colorbar(im, ax=axes[0], label="sample coverage")
    smart.Mosaic([t["frame"] for t in tiles.values()], [t["pos"] for t in tiles.values()],
                 cal, downsample=3).show(axes[1])
    fig.tight_layout(); handle.update(fig); plt.close(fig)


try:
    while queue and len(tiles) < MAX_TILES:
        i, j = queue.popleft()
        tx, ty = x0 + i * pitch, y0 + j * pitch
        if not box.contains(tx, ty):
            continue
        move(tx - here[0], ty - here[1])
        here = (tx, ty)
        frame = snap()
        score = empty_ref.coverage(frame)
        tiles[(i, j)] = {"pos": smart.get_xy(client, STAGE), "score": score, "frame": frame}
        if score >= COVERAGE_THRESHOLD:
            for di, dj in neigh:
                n = (i + di, j + dj)
                if n not in queued:
                    queued.add(n); queue.append(n)
        draw_map()
except KeyboardInterrupt:
    print("stopped")

move(x0 - here[0], y0 - here[1])
I = [k[0] for k in tiles]; J = [k[1] for k in tiles]
raster = (max(I) - min(I) + 1) * (max(J) - min(J) + 1)
n_sample = sum(t["score"] >= COVERAGE_THRESHOLD for t in tiles.values())
print(f"imaged {len(tiles)} tiles ({n_sample} with sample); a raster over the same bounding box "
      f"would need {raster}. {'Stopped at MAX_TILES. ' if queue else ''}Back at the start.")

## 6 B: Detect objects and image them in detail

Objects are found in every sample tile and converted to stage coordinates. Detections of the same
object in overlapping tiles are merged. The `N_TARGETS` largest are then visited one by one.

In [ ]:
DETECT = dict(min_area=40, max_area=5000, k=3.0, dark_objects=False)
MERGE_UM = 15                 # detections closer than this are one object
N_TARGETS = 6
AUTOFOCUS = False             # server autofocus at every target (moves Z within ±AF_RANGE_UM)
AF_RANGE_UM, AF_STEP_UM = 30, 3
DETAIL_EXPOSURE_MS = None     # e.g. 200 for a longer exposure on the targets; None = unchanged
SAVE_TO_IMSWITCH = True       # write the detail images to the ImSwitch data folder

found = []
for t in tiles.values():
    if t["score"] < COVERAGE_THRESHOLD:
        continue
    for o in smart.find_objects(t["frame"], **DETECT):
        ox, oy = cal.stage_offset_of_pixel(*o["offset"])
        found.append({"x": t["pos"][0] + ox, "y": t["pos"][1] + oy, "area": o["area"]})
targets = []
for f in sorted(found, key=lambda f: -f["area"]):
    if all(np.hypot(f["x"] - g["x"], f["y"] - g["y"]) > MERGE_UM for g in targets):
        targets.append(f)
print(f"{len(found)} detections -> {len(targets)} objects; imaging the {min(N_TARGETS, len(targets))} largest")

m = smart.Mosaic([t["frame"] for t in tiles.values()], [t["pos"] for t in tiles.values()], cal, downsample=2)
fig, ax = plt.subplots(figsize=(9, 7))
m.show(ax)
for k, g in enumerate(targets):
    r, c = m.pixel_of_stage(g["x"], g["y"])
    ax.plot(c, r, "o", mfc="none", mec="r" if k < N_TARGETS else "y", ms=10)
    if k < N_TARGETS:
        ax.text(c + 6, r, str(k), color="r")
plt.show()

In [ ]:
detector = client.settingsManager.getDetectorNames()[0]
old_exposure = client.settingsManager.getDetectorParameter("exposure", detector)
details = []
here = smart.get_xy(client, STAGE)
start = here
try:
    if DETAIL_EXPOSURE_MS:
        client.settingsManager.setDetectorExposureTime(detector, DETAIL_EXPOSURE_MS)
    for k, g in enumerate(targets[:N_TARGETS]):
        dx, dy, clipped = box.limit(here[0], here[1], g["x"] - here[0], g["y"] - here[1])
        if clipped:
            print(f"target {k} is outside the safety box - skipped")
            continue
        move(dx, dy)
        here = (here[0] + dx, here[1] + dy)
        if AUTOFOCUS:
            st = client.autofocusManager.autoFocusAndWait(rangez=AF_RANGE_UM, resolutionz=AF_STEP_UM)
            print(f"target {k}: focus at Z={st.get('currentZ')}")
        img = snap()
        saved = client.recordingManager.snapImageToPath(f"target_{k:02d}") if SAVE_TO_IMSWITCH else {}
        details.append({"k": k, "pos": here, "image": img, "file": saved.get("fullPath")})
finally:
    if DETAIL_EXPOSURE_MS and old_exposure:
        client.settingsManager.setDetectorExposureTime(detector, old_exposure)
    move(start[0] - here[0], start[1] - here[1])

if details:
    fig, axes = plt.subplots(1, len(details), figsize=(3.2 * len(details), 3), squeeze=False)
    for ax, d in zip(axes[0], details):
        ax.imshow(d["image"], cmap="gray"); ax.set_title(f"target {d['k']}"); ax.axis("off")
    plt.show()
for d in details:
    print(d["k"], np.round(d["pos"], 1), d["file"])

## 7 C: Event-driven time-lapse

Watches the current field of view every `SLOW_INTERVAL_S`. When the frame differs from the
previous one by more than `CHANGE_THRESHOLD` (1 − correlation), it records a burst of `BURST_N`
frames as fast as the camera allows and saves them. Try it by moving the stage briefly in the UI,
or by moving something on the sample.

In [ ]:
SLOW_INTERVAL_S = 5
CHANGE_THRESHOLD = 0.15
BURST_N = 10
DURATION_S = 120
SAVE_TO_IMSWITCH = True


def change(a, b):
    a = (a - a.mean()) / (a.std() + 1e-6); b = (b - b.mean()) / (b.std() + 1e-6)
    return float(1 - (a * b).mean())


log, events = [], []
prev = snap()
t0 = time.time()
handle = display(None, display_id=True)
try:
    while time.time() - t0 < DURATION_S:
        time.sleep(SLOW_INTERVAL_S)
        frame = snap()
        c = change(prev, frame)
        log.append((time.time() - t0, c))
        if c > CHANGE_THRESHOLD:
            burst = []
            for n in range(BURST_N):
                if SAVE_TO_IMSWITCH:
                    client.recordingManager.snapImageToPath(f"event{len(events):03d}_{n:02d}")
                burst.append(snap())
            events.append({"t": time.time() - t0, "change": c, "frames": burst})
            frame = burst[-1]
        prev = frame
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
        axes[0].plot(*zip(*log), ".-"); axes[0].axhline(CHANGE_THRESHOLD, ls="--", c="r")
        for e in events:
            axes[0].axvline(e["t"], c="orange", alpha=0.5)
        axes[0].set_xlabel("time [s]"); axes[0].set_ylabel("change")
        axes[0].set_title(f"{len(events)} events, {len(log) + sum(len(e['frames']) for e in events)} frames")
        axes[1].imshow(frame, cmap="gray"); axes[1].axis("off")
        fig.tight_layout(); handle.update(fig); plt.close(fig)
except KeyboardInterrupt:
    print("stopped")
fixed_fast = int(DURATION_S / max(SLOW_INTERVAL_S / BURST_N, 0.1))
print(f"{len(events)} events. Frames taken: {len(log) + sum(len(e['frames']) for e in events)}, "
      f"versus ~{fixed_fast} at the burst rate throughout.")

Switch the light off when you are done:

In [ ]:
light(False)